# Pico – large pre-training on Google Colab

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Everything important (prepared data, checkpoints) is saved to Google Drive in `MyDrive/pico/`.
3. **If Colab disconnects:** reconnect and run all cells again. Data is copied back from Drive and training resumes from the last checkpoint (at most ~10 minutes lost).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Connect Google Drive

In [ ]:
import os, shutil
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/pico'
os.makedirs(DRIVE, exist_ok=True)

## 3. Get the code

In [ ]:
if not os.path.exists('/content/Pico'):
    !git clone https://github.com/Abhinav-Krishna-P/Pico-Next_Token_Predictor.git /content/Pico
%cd /content/Pico
!git pull
!pip install -q sentencepiece datasets

## 4. Prepare the data (first time ~30–60 min, later sessions ~1 min)

Downloads full TinyStories + 300k FineWeb-Edu documents, trains a 16k tokenizer and tokenizes everything (~965M tokens). The result is stored on Drive so this only happens once.

In [ ]:
DATA = '/content/data_big'
files = ['train.bin', 'val.bin', 'tokenizer.model', 'meta.json']
if os.path.exists(f'{DRIVE}/data_big/train.bin'):
    os.makedirs(DATA, exist_ok=True)
    for f in files:
        shutil.copy(f'{DRIVE}/data_big/{f}', DATA)   # local disk is much faster than Drive for training
    print('Copied prepared data from Drive')
else:
    !python prepare_data.py --out-dir {DATA}
    os.makedirs(f'{DRIVE}/data_big', exist_ok=True)
    for f in files:
        shutil.copy(f'{DATA}/{f}', f'{DRIVE}/data_big/')
    print('Saved prepared data to Drive')

## 5. Pre-train (~34M parameters, 600M tokens, roughly 4–6 hours on a T4)

Prints progress every 10 steps, and every 250 steps the validation loss plus two sample texts. A checkpoint is saved to Drive every 10 minutes. If the session ends, re-run all cells and it continues where it stopped.

Optional: add `--compile` for extra speed (takes ~1 min to start).

In [ ]:
!python pretrain.py --data-dir {DATA} --out {DRIVE}/pico_big.pt

## 6. Try the pre-trained model

In [ ]:
import torch, sentencepiece as spm
from model import Pico
ckpt = torch.load(f'{DRIVE}/pico_big.pt', map_location='cuda')
sp = spm.SentencePieceProcessor(model_file=f'{DRIVE}/pico_big_tokenizer.model')
model = Pico(**ckpt['config']).cuda().eval()
model.load_state_dict(ckpt['model'])
print(f"step {ckpt['step']}, val loss {ckpt['val_loss']:.4f}")
print()
for prompt in ['Once upon a time', 'The history of science', 'My favourite food is']:
    idx = torch.tensor([sp.encode(prompt)], device='cuda')
    print(sp.decode(model.generate(idx, max_new_tokens=100, stop_id=sp.eos_id())[0].tolist()))
    print()

## 7. Fine-tune for grammar correction

First upload your `train.jsonl` to **Google Drive → MyDrive/pico/train.jsonl**. With the 512-token context ~97% of the pairs fit; 6000 steps ≈ one pass over them.

In [ ]:
import os
if not os.path.exists('/content/synthetic.jsonl'):
    !python make_synthetic.py --out /content/synthetic.jsonl --pairs 400000
if not os.path.exists('/content/synthetic.jsonl'):
    raise RuntimeError('make_synthetic.py failed (see the error above), not fine-tuning')
!python finetune_grammar.py --data {DRIVE}/train.jsonl /content/synthetic.jsonl --base {DRIVE}/pico_big.pt --out {DRIVE}/pico_big_grammar.pt --tokenizer-out {DRIVE}/pico_big_grammar_tokenizer.model --steps 15000 --batch-size 32


## 8. Correct sentences

In [ ]:
for s in ['they was playing in park', 'She go to school yesterday.', 'I has a apple.',
          'He dont like coffee.', 'I am work as software engineer.']:
    print('IN :', s)
    !python correct.py --model {DRIVE}/pico_big_grammar.pt "{s}"
    print()

## 9. Use it on your laptop

Download `pico_big_grammar.pt` and `pico_big_grammar_tokenizer.model` from `MyDrive/pico/` into the Pico project folder, then:

```
python correct.py --model pico_big_grammar.pt "they was playing in park"
```